# 00 · Setup: Drive, package, GPU, paths, manifests

**GitHub = code · Colab `/content` = fast ephemeral execution · Google Drive (`MyDrive/GEMS_2`) = persistent data and artifacts.**

Run this notebook at the start of every Colab session (or copy its first cell into any other notebook: every notebook here starts with the same idempotent bootstrap).

1. Mount Drive · 2. clone the repository at a pinned commit and `pip install -e .[colab]` · 3. check PyTorch / GPU · 4. record package version + commit · 5. free disk / VRAM · 6. validate data paths · 7. create the workspace folders · 8. stage the core data from Drive to `/content/gems_work` · 9. reload experiment manifests · 10. environment report.

Nothing here trains a model. No credentials are stored in the notebook: a private repository is cloned with a token read from Colab *Secrets* (`GITHUB_TOKEN`).

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'          # GitHub owner/name of this project
COMMIT = 'main'                    # pin a commit SHA for reproducible experiments
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')       # editable install is not visible to an already-running kernel
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
print(subprocess.run(['git', '-C', '/content/GEMS', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout)

## Download and organize the data (once)

DrivenData links are **personal and expire**. Log in, open the competition's *Data download* page, copy each file's link and paste it into `DATA` below. If the data comes as one zip, paste its link into `ARCHIVE_URL` instead.

Each file is fetched with `wget -c` (resumable) to the fast local disk. It is then checked (it must open as a GeoTIFF, and its size is compared with the known release), moved into the workspace layout (`data/raw/geodawn`, `data/raw/faults`) and copied to Drive with a checksum.

Files already on Drive are skipped, so re-running this cell is harmless. Never commit or share the links.

The second cell optionally lists the USGS 1 m DEM products for the study area with `wget` (metadata only; no tiles).

The legacy Phase 1–4 `outputs/` (feature caches, folds, model maps) are not downloadable: upload them once from the local project (`docs/GOOGLE_DRIVE_LAYOUT.md`).

In [ ]:
# --- download + organize the competition data (wget, resumable) -----------------------------------
import shutil, subprocess, zipfile
from pathlib import Path
LOCAL = Path(colab.LOCAL_ROOT)

# workspace path -> (download URL pasted from DrivenData, expected bytes or None)
DATA = {
    'data/raw/geodawn/gems-geodawn-numerical-features.tif': ('', 418_912_844),
    'data/raw/faults/existing_faults.tif':                  ('', 425_830),
    'data/raw/geodawn/example_submission.tif':              ('', 1_599_597),
}
ARCHIVE_URL = ''      # alternative: one zip with the three files (any folder structure inside)

def wget(url, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '-c', '-q', '--show-progress', '--tries=5', '--timeout=60', '--retry-connrefused',
                    '-O', str(dst), url], check=True)

def check_tif(p, expected):
    import rasterio
    with rasterio.open(p) as src:                    # fails loudly on an HTML error page / truncated file
        info = f'{src.count} band(s), {src.width} x {src.height}, {src.crs}'
    if expected and p.stat().st_size != expected:
        print(f'  WARNING {p.name}: {p.stat().st_size} bytes, expected {expected} (new data release?)')
    return info

todo = {rel: v for rel, v in DATA.items() if not (Path(DRIVE_ROOT) / rel).exists()}
for rel in DATA:
    if rel not in todo:
        print('on Drive ', rel)

# 1) optional archive -> data/raw/_incoming -> known file names moved into the layout
if todo and ARCHIVE_URL:
    inc = LOCAL / 'data/raw/_incoming'
    zp = inc / 'gems_data.zip'
    wget(ARCHIVE_URL, zp)
    with zipfile.ZipFile(zp) as z:
        z.extractall(inc)
    for rel in todo:
        hits = [p for p in inc.rglob(Path(rel).name) if p.is_file()]
        if hits and not (LOCAL / rel).exists():
            (LOCAL / rel).parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(hits[0]), str(LOCAL / rel))

# 2) individual files: wget -> <name>.partial -> check -> rename into place
for rel, (url, expected) in todo.items():
    dst = LOCAL / rel
    if not dst.exists():
        if not url:
            print('NO URL   ', rel)
            continue
        part = dst.with_name(dst.name + '.partial')
        wget(url, part)
        check_tif(part, expected)
        part.replace(dst)
    print('ready    ', rel, '|', check_tif(dst, expected))

# 3) copy to Drive through .partial + sha256 check + atomic rename (local copies stay as the working set)
ready = [rel for rel in todo if (LOCAL / rel).exists()]
if ready:
    colab.sync().push(ready, verify='sha256')
shutil.rmtree(LOCAL / 'data/raw/_incoming', ignore_errors=True)

In [ ]:
# Optional: USGS TNM 1 m DEM URL list for the study area via the TNM Access API (metadata only: NO tiles here;
# tiles are downloaded later by notebook 05 after explicit approval). Skip if you already have your own JSON.
FETCH_DEM_URL_LIST = False
dem_json = Path(DRIVE_ROOT) / 'data/raw/dem_1m/tnm_urls.json'
if FETCH_DEM_URL_LIST and not dem_json.exists():
    import json, urllib.parse
    import rasterio
    from rasterio.warp import transform_bounds
    with rasterio.open(LOCAL / 'data/raw/geodawn/gems-geodawn-numerical-features.tif') as src:
        w, s, e, n = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
    items, offset = [], 0
    while True:
        q = urllib.parse.urlencode({'datasets': 'Digital Elevation Model (DEM) 1 meter', 'prodFormats': 'GeoTIFF',
                                    'bbox': f'{w:.4f},{s:.4f},{e:.4f},{n:.4f}', 'max': 500, 'offset': offset})
        page = LOCAL / f'data/raw/dem_1m/_tnm_page_{offset}.json'
        page.unlink(missing_ok=True)                  # a page must never be "continued" by wget -c
        wget('https://tnmaccess.nationalmap.gov/api/v1/products?' + q, page)
        r = json.loads(page.read_text())
        items += r.get('items', [])
        offset += 500
        if offset >= int(r.get('total', 0)) or not r.get('items'):
            break
    out = LOCAL / 'data/raw/dem_1m/tnm_urls.json'
    out.write_text(json.dumps({'items': items, 'bbox_wgs84': [w, s, e, n]}, indent=1))
    for p in out.parent.glob('_tnm_page_*.json'):
        p.unlink()
    gib = sum(i.get('sizeInBytes') or 0 for i in items) / 2**30
    print(f'{len(items)} products listed ({gib:.1f} GiB if ALL were downloaded - the pilot uses ~24 tiles)')
    colab.sync().push(['data/raw/dem_1m/tnm_urls.json'])
elif dem_json.exists():
    print('DEM URL list on Drive:', dem_json)

In [ ]:
# PyTorch / GPU and the auto-configuration the trainers will use
from gems_bench.training import gpu
g = gpu.detect()
print(g)
print('autoconfig (256 px, effective batch 16, 29 channels):', gpu.autoconfig(g, patch_px=256, effective_batch=16, n_channels=29))

In [ ]:
# Drive layout: create missing workspace folders on Drive and locally (never deletes anything)
from pathlib import Path
from gems_bench.paths import WORKSPACE_DIRS
for d in WORKSPACE_DIRS:
    (Path(DRIVE_ROOT) / d).mkdir(parents=True, exist_ok=True)
    (Path(colab.LOCAL_ROOT) / d).mkdir(parents=True, exist_ok=True)
required = ['data/raw/geodawn/gems-geodawn-numerical-features.tif', 'data/raw/faults/existing_faults.tif',
            'data/raw/geodawn/example_submission.tif', 'outputs/benchmark/folds', 'outputs/phase4/experiments']
for r in required:
    print(('OK      ' if (Path(DRIVE_ROOT) / r).exists() else 'MISSING ') + r)
print('See docs/GOOGLE_DRIVE_LAYOUT.md for what to upload once from the local project.')

In [ ]:
# Stage the core data to the fast local disk (unchanged files are skipped on later sessions)
staged = colab.stage_core()
staged.groupby('action').bytes.agg(['count', 'sum'])

In [ ]:
# Existing experiment manifests and the model registry (from Drive)
import json
import pandas as pd
from gems_bench.artifacts import ModelRegistry
rows = []
for m in sorted(Path(DRIVE_ROOT, 'experiments').glob('*/manifest.json')):
    j = json.loads(m.read_text())
    rows.append({'experiment': j['experiment_id'], 'kind': j['kind'], 'commit': j.get('git_commit', '')[:10],
                 **{s: v['status'] for s, v in j['stages'].items()}})
display(pd.DataFrame(rows))
display(ModelRegistry(Path(DRIVE_ROOT) / 'model_registry').table())

In [ ]:
colab.environment_report()